In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
from sklearn.metrics import classification_report, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from joblib import Parallel, delayed
import multiprocessing


In [2]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       
        self.threshold = threshold   
        self.left = left             
        self.right = right          
        self.value = value           

    def is_leaf(self):
        return self.value is not None

Дерево для классификации

In [3]:
def gini_impurity(y):
    m = len(y)
    if m == 0:
        return 0
    counts = np.bincount(y)
    probabilities = counts / m
    return 1.0 - np.sum(probabilities ** 2)

def information_gain(y, left_y, right_y):
    p = len(left_y) / len(y)
    return gini_impurity(y) - p * gini_impurity(left_y) - (1 - p) * gini_impurity(right_y)


In [4]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_split):
            most_common = np.bincount(y).argmax()
            return Node(value=most_common)

        best_feat, best_thresh, best_gain = None, None, -1
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                left_idx = np.where(X_column <= threshold)[0]
                right_idx = np.where(X_column > threshold)[0]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                    
                gain = information_gain(y, y[left_idx], y[right_idx])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = threshold

        if best_gain <= 0:
            return Node(value=np.bincount(y).argmax())

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
            
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


In [5]:
data = pd.read_csv('../lessons/Dry_Bean_Dataset.csv', delimiter=';', decimal=',')
print(data.columns.tolist())
data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)
X = data.drop(['Class'], axis=1).values
le = LabelEncoder()
y = le.fit_transform(data['Class'].values)
#y = data['Class'].values
#y_decoded = le.inverse_transform(y_encoded)  

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

print(pd.Series(y).value_counts())

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
Признаков: 16, Объектов: 13611
3    3546
6    2636
5    2027
4    1928
2    1630
0    1322
1     522
Name: count, dtype: int64


In [6]:
model = DecisionTreeClassifier( )
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = classification_report(y_test, y_pred)
print(accuracy)

              precision    recall  f1-score   support

           0       0.91      0.87      0.89       261
           1       1.00      1.00      1.00       117
           2       0.89      0.93      0.91       317
           3       0.88      0.90      0.89       671
           4       0.95      0.93      0.94       408
           5       0.95      0.91      0.93       413
           6       0.83      0.86      0.85       536

    accuracy                           0.90      2723
   macro avg       0.92      0.91      0.92      2723
weighted avg       0.90      0.90      0.90      2723



Дерево для регрессии

In [7]:
def mse_impurity(y):
    m = len(y)
    if m == 0:
        return 0
    diff = y - np.mean(y)
    return np.sum(diff ** 2) / m

def information_gain(y, left_y, right_y):
    p = len(left_y) / len(y)
    return mse_impurity(y) - p * mse_impurity(left_y) - (1 - p) * mse_impurity(right_y)


In [8]:
class DecisionTreeRegration:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build_tree(X, y)
        return self

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        
        if depth >= self.max_depth or n_samples < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(value=np.mean(y))

        best_feat, best_thresh, best_gain = None, None, -1
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for threshold in thresholds:
                left_idx = np.where(X_column <= threshold)[0]
                right_idx = np.where(X_column > threshold)[0]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                    
                gain = information_gain(y, y[left_idx], y[right_idx])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = threshold

        if best_gain <= 0 or best_feat is None:
            return Node(value=np.mean(y))

        left_idx = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idx = np.where(X[:, best_feat] > best_thresh)[0]
        
        left_child = self._build_tree(X[left_idx, :], y[left_idx], depth + 1)
        right_child = self._build_tree(X[right_idx, :], y[right_idx], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def apply(self, X):
        return np.array([self._get_leaf_index(x, self.root) for x in X])

    def _get_leaf_index(self, x, node):
        if node.is_leaf():
            return id(node)
        if x[node.feature] <= node.threshold:
            return self._get_leaf_index(x, node.left)
        return self._get_leaf_index(x, node.right)

In [9]:
data = pd.read_csv('../lessons/hour.csv')
X = data.drop(['cnt','instant','dteday'], axis=1).values
y = data['cnt'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")




Признаков: 14, Объектов: 17379


In [10]:
model = DecisionTreeRegration( )
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = r2_score(y_test, y_pred)
print(accuracy)

0.9986419015216151


Ансамблевые методы классификация

In [11]:
df = pd.read_csv('../lessons/bank-additional-full.csv', delimiter=';', decimal='.')
print(df.columns.tolist())
data = pd.get_dummies(df, columns=["job","marital","education","default","housing","loan","contact","month","day_of_week",'poutcome',])

data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)
X = data.drop(['y'], axis=1).values
le = LabelEncoder()
y = le.fit_transform(data['y'].values)
#y = data['Class'].values
#y_decoded = le.inverse_transform(y_encoded)  

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

print(pd.Series(y).value_counts())





X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
Признаков: 63, Объектов: 41188
0    36548
1     4640
Name: count, dtype: int64


In [12]:
class RandomForestClisifier:
    def __init__(self, n_estimators=100, max_depth = 3):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.trees = []

    def _train_single_tree(self, X_train, y_train, random_state):
        X_boot, y_boot = resample(X_train, y_train, replace=True, random_state=random_state)
        model = DecisionTreeClassifier(max_depth=self.max_depth)
        model.fit(X_boot, y_boot)
        return model

    def fit(self, X_train, y_train):
        n_jobs = multiprocessing.cpu_count()
        
        self.trees = Parallel(n_jobs=n_jobs)(
            delayed(self._train_single_tree)(X_train, y_train, i)
            for i in range(self.n_estimators)
        )
        return self

    def predict(self, X_test):
        predicts = []
        for model in self.trees:
            y_pred = model.predict(X_test)
            predicts.append(y_pred)
        predicts = np.array(predicts).T
        return np.array( [np.bincount(arr).argmax() for arr in predicts] )
            

    def score(self,y_test, y_pred):
        accuracy = classification_report(y_test, y_pred)
        print(accuracy)


In [13]:
model = RandomForestClisifier(n_estimators=10, max_depth=4)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


model.score(y_test, y_pred)


              precision    recall  f1-score   support

           0       0.94      0.97      0.95      7310
           1       0.66      0.52      0.58       928

    accuracy                           0.92      8238
   macro avg       0.80      0.74      0.77      8238
weighted avg       0.91      0.92      0.91      8238



In [14]:
class GBMClassifier:
    def __init__(self, logitboost=False, learning_rate=0.1, n_estimators=100,
                 max_depth=3, random_state=0):
        self.logitboost = logitboost
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.K = None
        self.trees = None

    def _softmax(self, predictions):
        exp = np.exp(predictions - np.max(predictions, axis=1, keepdims=True))
        return exp / np.sum(exp, axis=1, keepdims=True)

    def _compute_gammas(self, residuals, leaf_indexes):
        gammas = np.zeros_like(residuals)
        for leaf in np.unique(leaf_indexes):
            mask = leaf_indexes == leaf
            gammas[mask] = np.mean(residuals[mask])
        return gammas

    def fit(self, X, y):
        np.random.seed(self.random_state)
        y = y.astype(int)
        self.K = len(np.unique(y))
        self.trees = {k: [] for k in range(self.K)}
        
        one_hot_y = np.eye(self.K)[y]
        predictions = np.zeros((len(X), self.K))
        
        for epoch in range(self.n_estimators):
            probabilities = self._softmax(predictions)
            
            for k in range(self.K):
                if self.logitboost:
                    p = probabilities[:, k]
                    y_k = one_hot_y[:, k]
                    residuals = (y_k - p) / (p * (1 - p) + 1e-8)
                    residuals = residuals * (self.K - 1) / self.K
                else:
                    residuals = one_hot_y[:, k] - probabilities[:, k]
                
                tree = DecisionTreeRegration(max_depth=self.max_depth)
                tree.fit(X, residuals)
                self.trees[k].append(tree)
                
                leaf_indexes = tree.apply(X)
                gammas = self._compute_gammas(residuals, leaf_indexes)
                predictions[:, k] += self.learning_rate * gammas
        
        return self

    def predict(self, X):
        predictions = np.zeros((len(X), self.K))
        
        for epoch in range(self.n_estimators):
            for k in range(self.K):
                tree = self.trees[k][epoch]
                predictions[:, k] += self.learning_rate * tree.predict(X)
        
        return np.argmax(predictions, axis=1)

    def score(self,y_test, y_pred):
            accuracy = classification_report(y_test, y_pred)
            print(accuracy)

In [15]:
model = GBMClassifier(n_estimators=10, max_depth=4)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


model.score(y_test, y_pred)


              precision    recall  f1-score   support

           0       0.94      0.97      0.95      7310
           1       0.66      0.51      0.58       928

    accuracy                           0.91      8238
   macro avg       0.80      0.74      0.76      8238
weighted avg       0.91      0.91      0.91      8238



Ансамблевые методы для регресссии

In [16]:
class RandomForestRegration:
    def __init__(self, n_estimators=100, max_depth = 3):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.trees = []

    def _train_single_tree(self, X_train, y_train, random_state):
        X_boot, y_boot = resample(X_train, y_train, replace=True, random_state=random_state)
        model = DecisionTreeRegration(max_depth=self.max_depth)
        model.fit(X_boot, y_boot)
        return model

    def fit(self, X_train, y_train):
        n_jobs = multiprocessing.cpu_count()
        
        self.trees = Parallel(n_jobs=n_jobs)(
            delayed(self._train_single_tree)(X_train, y_train, i)
            for i in range(self.n_estimators)
        )
        return self

    def predict(self, X_test):
        predicts = []
        for model in self.trees:
            y_pred = model.predict(X_test)
            predicts.append(y_pred)
        predicts = np.array(predicts).T
        return np.array( [np.mean(arr) for arr in predicts] )
            

    def score(self,y_test, y_pred):
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print(f"r^2 : {r2}\n mae: {mae}")


In [17]:
data = pd.read_csv('../lessons/hour.csv')
X = data.drop(['cnt','instant','dteday'], axis=1).values
y = data['cnt'].values



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)
scaler_y = StandardScaler()
y_train= scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()


print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")




Признаков: 14, Объектов: 17379


In [18]:
model = RandomForestRegration(n_estimators=10, max_depth=4)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


model.score(y_test, y_pred)

r^2 : 0.980837440859187
 mae: 0.0923095773654875


In [ ]:
class GBMRegressor:
    def __init__(self, learning_rate=0.1, n_estimators=100, max_depth=3, random_state=0):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.trees = []
        self.initial_leaf = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        
        self.initial_leaf = np.mean(y)
        predictions = np.full(len(y), self.initial_leaf, dtype=float)

        for epoch in range(self.n_estimators):
            residuals = y - predictions
            
            tree = DecisionTreeRegration(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            predictions += self.learning_rate * tree.predict(X)

        return self

    def predict(self, X):
        predictions = np.full(len(X), self.initial_leaf, dtype=float)
        
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)
        
        return predictions

    def score(self, X_test, y_test):
        y_pred = self.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)
        print(f"R²: {r2:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        return r2

In [20]:
model = GBMRegressor(learning_rate=0.1, n_estimators=100, max_depth=3)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
model.score(X_test, y_test)

R²: 0.9991
MAE: 0.0182
RMSE: 0.0291


0.9991115597035657